# PIPELINE STEPS : VALIDATED STEPS ONLY

### Important for auto reloading any changes in files in VS CODE withouting restarting kernel

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

project_root = os.path.abspath("../..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

### For auto correcting execution path for notebook

In [ ]:
import os
print(os.getcwd())
import sys
import os

project_root = os.path.abspath("./..")

if project_root not in sys.path:
    sys.path.append(project_root)

print(project_root)

## Stage 1 : Imports

In [ ]:
from src_v2.parsers.docling_parser import (
    DoclingParser
)

from src_v2.parsers.markdown_section_parser import (
    MarkdownSectionParser
)
from src_v2.preprocessing.formula_cleaner import (
    FormulaCleaner
)
from src_v2.parsers.level_inference import (
    LevelInference
)
from src_v2.normalization.section_flattener import (
    SectionFlattener
)

## Stage 2: Parsing source documents using Docling

In [ ]:
docling=DoclingParser()
raw_markdown = docling.parse(
    "Transformer_Architecture_Guide.pdf"
)

print(
    len(raw_markdown.splitlines())
)

## Stage 3: Process (string :text in markdown format) obtained from Docling parsing for LATEX errors

In [ ]:
cleaned = FormulaCleaner().process_markdown(
    raw_markdown
)
print(
    "Cleaned lines:",
    len(cleaned.splitlines())
)

## Stage 4: Creating a document(markdown parser needs 2 inputs a string(markdown text) and A Document(random)

In [ ]:
from src_v2.models.document import Document

doc = Document(
    doc_id="transformer",
    title="Transformer Architecture Guide",
    source_type="pdf",
    source_path="/home/arun/Desktop/AIProjects/rag-evaluation-framework/experiments/Transformer_Architecture_Guide.pdf"
)

### This parser parses the string to remove any markdown format in the string

In [ ]:
parser = MarkdownSectionParser()

parsed_doc = parser.parse(
    cleaned,
    doc
)

## Stage 5: Level Inferencing constructs levels in a documents based on a set of rules

In [ ]:
inferencer = LevelInference()

parsed_doc = inferencer.infer(
    parsed_doc
)

## Stage 6: Hierarchy builder builds the hierarchy based on the levels we created

In [ ]:
from src_v2.parsers.hierarchy_builder import(HierarchyBuilder)
builder = HierarchyBuilder()

parsed_doc = builder.build(
    parsed_doc
)

#### For validation :Tetsing purposes

In [ ]:
for section in parsed_doc.sections:

    print(
        section.title,
        "| parent =", section.parent_section_id,
        "| children =", len(section.subsections)
    )

## Stage 7: Convert heirarchical document tree into flat list while preserving hierarchy information in the path(metadata)

In [ ]:
flattener = SectionFlattener()

flattened = flattener.flatten(
    parsed_doc
)

print(
    "Flattened sections:",
    len(flattened)
)

#### for validation :testing purposes

In [ ]:
for section in flattened:

    print("=" * 80)

    print(section.section_title)

    print(section.path)

## Stage 8: Section Chunker -Section aware + Structure (Tables/Formula) Aware Chunking

In [ ]:
from src_v2.chunking import (
    SectionChunker
)

chunker = SectionChunker()

chunks = chunker.chunk(
    document=document,
    sections=flattened_sections
)

print(
    f"{len(chunks)} chunks created"
)